In [3]:
import os
from dotenv import load_dotenv

# Create .env beforehand in project root directory and program it with the line: HF_Token = "Your Own Token Sent by Hugging Face"
load_dotenv()
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HF_Token")  # HF_Token: the keyword named in .env (see above)

from langchain_huggingface import HuggingFaceEndpointEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

def setup_retriever(pdf_path):
    loader = PyPDFLoader(pdf_path)
    doc = loader.load()

    # Split the loaded pdf file into chunks
    splitter = RecursiveCharacterTextSplitter(
        separators = ["\n\n", "\n", " ", ""],  # default separator list
        chunk_size=1000,  # max number of characters per chunk
        chunk_overlap=200)  # number of characters shared between consecutive chunks
    chunks = splitter.split_documents(doc)

    # Enable calling an embedding model on Hugging Face
    embeddings = HuggingFaceEndpointEmbeddings(
        model="BAAI/bge-small-en-v1.5",
        task="feature-extraction",  # the only task for class HuggingFaceEndpointEmbeddings
    )

    # Create a vector store using Chroma database and insert chunks to it
    vector_store = Chroma.from_documents(chunks, embeddings)

    # Create a retriever for the vector store
    retriever = vector_store.as_retriever(
        search_type = "similarity",
        search_kwargs = {"k" : 3},  # top 3 most similar documents
    )
    # NOTE: "retriever" is a configured object ready to accept queries (knowing none of specific questions at this time)

    return retriever

# Call the custom function "setup_retriever"; load an example pdf file -- the paper from https://arxiv.org/pdf/2502.00835
retriever = setup_retriever("https://arxiv.org/pdf/2502.00835")  # a local path or URL 

# Enable calling an llm on Hugging Face; model runs on remote servers (vs. local computer if using HuggingFacePipeline)
hf_end = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
)

# Wrap the object "hf_end" for an Instruct llm (see "repo_id" above)
model = ChatHuggingFace(llm=hf_end)

# Create a prompt with placeholders "input" and "context"
prompt = ChatPromptTemplate.from_messages(
    [{"role": "system", "content": "Answer based on the context: \n\n{context}"},
     {"role": "user", "content": "Please answer this question: {input}"}])

# Create chains using LangChain classes
combine_docs_chain = create_stuff_documents_chain(model, prompt)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)
# Explain the above two lines:
# 1. When retrieval_chain.invoke({"input": ...}) called (see below), the value of key "input" (default key name) passed to the object "retriever" -- That's when the similarity search actually happens.
# 2. The returned documents formatted into a string; then the string inserted into "prompt" whereever the default variable {context} appears
# 3. The now-complete prompt sent to "model" as a single request

while True:
    question = input("Question: ")
    if question.lower() == "exit":
        break

    answer = retrieval_chain.invoke({"input": question})
    
    print(f"Answer: {answer["answer"]}\n\n")

Question:  What's CAIMAN?


Answer: Based on the provided text, CAIMAN appears to be a method or algorithm for controlling or manipulating objects, particularly in a robotic or artificial intelligence context. It seems to be designed for handling complex objects with asymmetric dynamics, and it is used for tasks such as object pushing or manipulation.




Question:  What's the latest progress on CAIMAN?


Answer: Based on the provided text, the latest progress on CAIMAN is its ability to learn and capture complex dynamics, particularly with irregular objects. The results from the cart pushing environment demonstrate CAIMAN's capabilities, and the text suggests that CAIMAN's advantages over other baselines would be further emphasized if obstacles, such as walls, were added to the environment. Additionally, the text proposes that CAIMAN's dynamics residual can be reused across different tasks if the underlying dynamics remain consistent, which could improve sample efficiency and enhance the exploration signal in the early stages of training.




Question:  What are other frameworks competing with CAIMAN?


Answer: Based on the provided text, the authors mention a few competitive baselines that are not explicitly named, but in the reference section [1], they cite a paper that includes the names of several researchers who have worked on a dynamics model, T. Miki, J. Lee, J. Hwangbo, L. Wellhausen, V. Koltun, and M. Hutter. However, this does not explicitly mention the name of the competing frameworks.

In the field of robotics and machine learning, there are several popular frameworks that could be competing with CAIMAN, such as:

1. ROS (Robot Operating System): A widely used framework for building robot applications.
2. PyRobot: A Python-based framework for building robot applications.
3. OpenCV: A computer vision library that provides a framework for building computer vision applications.
4. PPO (Proximal Policy Optimization): A popular reinforcement learning algorithm that can be used as a framework for building robotic applications.
5. PyTorch: A popular deep learning framework that c

Question:  Who are the authors of the paper "CAIMAN: Causal Action Influence Detection ..."?


Answer: The authors of the paper "CAIMAN: Causal Action Influence Detection for Sample-Efficient Loco-Manipulation" are:

1. Yuanchen Yuan
2. Jin Cheng
3. Núria Armengol Urpí
4. Stelian Coros




Question:  exit
